# Equivalent-resistance solver investigation

This notebook audits the bubble equivalent-resistance least-squares solves written by the collapse experiments. It separates experiment completion from solver convergence, identifies affected transitions, and provides a focused diagnostic for one selected bubble.

The equivalent-resistance solve fixes the upstream normalized head to 1 and the downstream head to 0. Internal node heads are optimized so that flow continuity residuals approach zero. A numerical `K_eq` can exist even when the optimizer did not converge; those rows must remain flagged as unsuccessful.

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'network_variants').is_dir() and (candidate / 'hierarchy_level_definition').is_dir():
            return candidate.resolve()
    raise RuntimeError('Could not locate the river-hierarchy project root.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_ROOT = PROJECT_ROOT / 'network_variants' / 'output'
PROJECT_ROOT, OUTPUT_ROOT.resolve()

(PosixPath('/Users/6256481/Code/river-hierarchy'),
 PosixPath('/Volumes/PhD/river_hierarchy/network_variants/output'))

## 1. Select experiments and one detailed case

The default audit covers the newly rerun examples 02 and 03. Change the lists when more examples have been completed. The selected detailed case defaults to the persistent failed base solve in example 02.

In [ ]:
EXAMPLE_IDS = ['02', '03']
EXPERIMENT_KINDS = ['indep', 'compound_bubbles']
CORRIDOR_MODES = ['anchored']

EXPERIMENT_NAMES = [
    f'{example_id}_{kind}__{corridor}'
    for example_id in EXAMPLE_IDS
    for kind in EXPERIMENT_KINDS
    for corridor in CORRIDOR_MODES
]

SELECTED_EXPERIMENT = '03_indep__anchored'
SELECTED_STATE_ID = 'S000_base'
SELECTED_BUBBLE_ID = 5
RUN_SELECTED_RESOLVES = True

missing_experiments = [name for name in EXPERIMENT_NAMES if not (OUTPUT_ROOT / name).is_dir()]
assert not missing_experiments, f'Missing experiment directories: {missing_experiments}'
EXPERIMENT_NAMES

['02_indep__anchored',
 '02_compound_bubbles__anchored',
 '03_indep__anchored',
 '03_compound_bubbles__anchored']

## 2. Load every state-level equivalent-resistance result

Each row below represents one compound bubble evaluated in one network state. `solver_success=False` means the least-squares termination criteria and the configured residual acceptance test were not satisfied.

In [14]:
def load_experiment_bubble_metrics(experiment_name):
    experiment_dir = OUTPUT_ROOT / experiment_name
    registry = pd.read_csv(experiment_dir / 'state_registry.csv')
    manifest = json.loads((experiment_dir / 'experiment_manifest.json').read_text())
    frames = []
    for row in registry.itertuples(index=False):
        metrics_path = Path(row.hierarchy_output_dir) / 'bubble_equivalent_metrics.csv'
        if not metrics_path.exists():
            continue
        frame = pd.read_csv(metrics_path)
        frame.insert(0, 'state_id', row.state_id)
        frame.insert(0, 'experiment_name', experiment_name)
        frame.insert(0, 'corridor_mode', experiment_name.rsplit('__', 1)[-1])
        frame.insert(0, 'experiment_kind', 'compound_bubbles' if 'compound_bubbles' in experiment_name else 'indep')
        frame.insert(0, 'example_id', manifest.get('example_id'))
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

bubble_metrics = pd.concat(
    [load_experiment_bubble_metrics(name) for name in EXPERIMENT_NAMES],
    ignore_index=True,
)
bubble_metrics['solver_success'] = bubble_metrics['solver_success'].fillna(False).astype(bool)
bubble_metrics['residual_relative'] = pd.to_numeric(bubble_metrics['residual_relative'], errors='coerce')
bubble_metrics.shape

(776, 27)

In [15]:
experiment_summary = (
    bubble_metrics.groupby(['example_id', 'experiment_kind', 'corridor_mode'], dropna=False)
    .agg(
        bubble_state_rows=('solver_success', 'size'),
        successful_solves=('solver_success', 'sum'),
        max_relative_residual=('residual_relative', 'max'),
        median_nfev=('solver_nfev', 'median'),
        max_nfev=('solver_nfev', 'max'),
    )
    .reset_index()
)
experiment_summary['failed_solves'] = experiment_summary['bubble_state_rows'] - experiment_summary['successful_solves']
experiment_summary['success_fraction'] = experiment_summary['successful_solves'] / experiment_summary['bubble_state_rows']
display(experiment_summary)

,example_id,experiment_kind,corridor_mode,bubble_state_rows,successful_solves,max_relative_residual,median_nfev,max_nfev,failed_solves,success_fraction
0,02,compound_bubbles,anchored,31,30,3.174711e-01,10.0,4000,1,0.967742
1,02,indep,anchored,161,150,3.176016e-01,10.0,4000,11,0.931677
2,03,compound_bubbles,anchored,113,113,2.258156e-14,0.0,46,0,1.000000
3,03,indep,anchored,471,470,8.895708e-02,0.0,4000,1,0.997877


In [16]:
fig = px.bar(
    experiment_summary,
    x='experiment_kind',
    y='failed_solves',
    color='corridor_mode',
    facet_col='example_id',
    barmode='group',
    hover_data=['bubble_state_rows', 'successful_solves', 'success_fraction', 'max_relative_residual'],
    title='Failed bubble solves by experiment',
)
fig.update_yaxes(matches=None)
fig.show()

## 3. Inspect all failed solves

The table is ordered by relative continuity residual, which is more comparable across bubbles than the absolute residual. Large bubbles and rows reaching 4,000 function evaluations are especially relevant to the current convergence problem.

In [18]:
failed_columns = [
    'example_id', 'experiment_name', 'state_id', 'compound_bubble_id', 'bubble_root_unit_id',
    'n_member_units', 'n_nodes', 'n_edges', 'solver_nfev', 'residual_norm',
    'residual_relative', 'Q_unit', 'K_eq', 'solver_message',
]
failed_solves = (
    bubble_metrics.loc[~bubble_metrics['solver_success'], failed_columns]
    .sort_values(['residual_relative', 'n_edges'], ascending=[False, False])
    .reset_index(drop=True)
)
display(failed_solves)

,example_id,experiment_name,state_id,compound_bubble_id,bubble_root_unit_id,n_member_units,n_nodes,n_edges,solver_nfev,residual_norm,residual_relative,Q_unit,K_eq,solver_message
0,02,02_indep__anchored,S011_unit_13,5,14,13,28,40,4000,963.934905,0.317602,3035.044467,1.085600e-07,The maximum number of function evaluations is ...
1,02,02_indep__anchored,S000_base,5,15,14,28,41,4000,964.193082,0.317471,3037.104989,1.084128e-07,The maximum number of function evaluations is ...
2,02,02_compound_bubbles__anchored,S000_base,5,15,14,28,41,4000,964.193082,0.317471,3037.104989,1.084128e-07,The maximum number of function evaluations is ...
3,02,02_indep__anchored,S023_unit_12,5,14,13,32,44,4000,964.203187,0.316702,3044.513070,1.078858e-07,The maximum number of function evaluations is ...
4,02,02_indep__anchored,S010_unit_9,5,14,13,30,42,4000,966.513129,0.300177,3219.809834,9.645829e-08,The maximum number of function evaluations is ...
5,02,02_indep__anchored,S014_unit_3,5,14,13,30,42,4000,883.790073,0.295993,2985.850455,1.121667e-07,The maximum number of function evaluations is ...
6,02,02_indep__anchored,S020_unit_8,5,14,13,31,43,4000,1062.669720,0.271316,3916.722863,6.518600e-08,The maximum number of function evaluations is ...
7,02,02_indep__anchored,S021_unit_15,5,13,13,30,42,4000,783.121733,0.217942,3593.261605,7.745016e-08,The maximum number of function evaluations is ...
8,02,02_indep__anchored,S009_unit_11,5,14,13,30,42,4000,185.595123,0.157857,1175.715815,7.234279e-07,The maximum number of function evaluations is ...
9,02,02_indep__anchored,S024_unit_14,5,14,13,30,42,4000,930.141110,0.144202,6450.261497,2.403507e-08,The maximum number of function evaluations is ...


In [19]:
plot_metrics = bubble_metrics.copy()
plot_metrics['solver_status'] = np.where(plot_metrics['solver_success'], 'success', 'failed')
plot_metrics['hover_id'] = (
    plot_metrics['experiment_name'].astype(str) + ' | ' +
    plot_metrics['state_id'].astype(str) + ' | bubble ' +
    plot_metrics['compound_bubble_id'].astype(str)
)
fig = px.scatter(
    plot_metrics,
    x='n_edges',
    y='residual_relative',
    color='solver_status',
    symbol='corridor_mode',
    facet_col='example_id',
    log_y=True,
    hover_name='hover_id',
    hover_data=['solver_nfev', 'n_nodes', 'Q_unit', 'K_eq'],
    color_discrete_map={'success': '#2ca02c', 'failed': '#d62728'},
    title='Continuity residual versus bubble size',
)
fig.show()

## 4. Identify transition comparisons affected by failed parent or child solves

These are the resistance-change comparisons that should not be interpreted until the associated solves converge or are explicitly excluded.

In [20]:
transition_frames = []
for experiment_name in EXPERIMENT_NAMES:
    path = OUTPUT_ROOT / experiment_name / 'equivalent_transition_registry.csv'
    if not path.exists():
        continue
    frame = pd.read_csv(path)
    frame.insert(0, 'experiment_name', experiment_name)
    transition_frames.append(frame)
equivalent_transitions = pd.concat(transition_frames, ignore_index=True)
for column in ['parent_solver_success', 'child_solver_success']:
    equivalent_transitions[column] = equivalent_transitions[column].fillna(False).astype(bool)
affected_transitions = equivalent_transitions.loc[
    ~(equivalent_transitions['parent_solver_success'] & equivalent_transitions['child_solver_success'])
].copy()
transition_summary = (
    equivalent_transitions.assign(
        both_success=lambda x: x.parent_solver_success & x.child_solver_success
    )
    .groupby('experiment_name')
    .agg(rows=('both_success', 'size'), both_success=('both_success', 'sum'))
    .reset_index()
)
transition_summary['affected_rows'] = transition_summary['rows'] - transition_summary['both_success']
display(transition_summary)
display(affected_transitions.head(100))

,experiment_name,rows,both_success,affected_rows
0,02_compound_bubbles__anchored,25,21,4
1,02_indep__anchored,155,129,26
2,03_compound_bubbles__anchored,98,98,0
3,03_indep__anchored,455,454,1


,experiment_name,transition_id,parent_state_id,child_state_id,match_method,parent_compound_bubble_id,child_compound_bubble_id,parent_bubble_root_unit_id,child_bubble_root_unit_id,parent_upstream_boundary_node,parent_downstream_boundary_node,child_upstream_boundary_node,child_downstream_boundary_node,parent_n_member_units,child_n_member_units,parent_solver_success,child_solver_success,K_before,K_after,G_before,G_after,Q_unit_before,Q_unit_after,residual_norm_before,residual_norm_after,residual_relative_before,residual_relative_after,Lambda_K,Lambda_G,percent_change_K,percent_change_G
4,02_indep__anchored,T001,S000_base,S001_unit_2,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,14.0,31.0,30.0,29.0,28.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
10,02_indep__anchored,T002,S000_base,S002_unit_18,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,15.0,31.0,30.0,31.0,30.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
15,02_indep__anchored,T003,S000_base,S003_unit_17,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,15.0,31.0,30.0,31.0,30.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
22,02_indep__anchored,T004,S000_base,S004_unit_5,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,14.0,31.0,30.0,29.0,28.0,14,13,False,True,1.084128e-07,3.780269e-06,9.224007e+06,2.645315e+05,3037.104989,514.326227,9.641931e+02,1.481186e-09,3.174711e-01,2.879858e-12,34.869223,0.028679,3386.922290,-97.132141
27,02_indep__anchored,T005,S000_base,S005_unit_25,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,15.0,31.0,30.0,31.0,30.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
33,02_indep__anchored,T006,S000_base,S006_unit_26,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,15.0,31.0,30.0,31.0,30.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
40,02_indep__anchored,T007,S000_base,S007_unit_1,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,14.0,31.0,30.0,29.0,28.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
46,02_indep__anchored,T008,S000_base,S008_unit_21,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,15.0,31.0,30.0,31.0,30.0,14,14,False,True,1.084128e-07,3.780418e-06,9.224007e+06,2.645210e+05,3037.104989,514.316047,9.641931e+02,4.986784e-10,3.174711e-01,9.695953e-13,34.870603,0.028677,3387.060328,-97.132255
52,02_indep__anchored,T009,S000_base,S009_unit_11,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,14.0,31.0,30.0,29.0,28.0,14,13,False,False,1.084128e-07,7.234279e-07,9.224007e+06,1.382308e+06,3037.104989,1175.715815,9.641931e+02,1.855951e+02,3.174711e-01,1.578571e-01,6.672904,0.149860,567.290420,-85.014021
58,02_indep__anchored,T010,S000_base,S010_unit_9,matched_parent_upstream_boundary_node__matched...,5.0,5.0,15.0,14.0,31.0,30.0,29.0,28.0,14,13,False,False,1.084128e-07,9.645829e-08,9.224007e+06,1.036718e+07,3037.104989,3219.809834,9.641931e+02,9.665131e+02,3.174711e-01,3.001771e-01,0.889732,1.123934,-11.026809,12.393407


## 5. Detailed selected-bubble diagnostic

This section loads the stored edge resistances and node heads for one bubble. The topology plot uses the saved node coordinates; edge hover text reports resistance, normalized flow, and head drop.

In [21]:
selected_experiment_dir = OUTPUT_ROOT / SELECTED_EXPERIMENT
selected_registry = pd.read_csv(selected_experiment_dir / 'state_registry.csv')
selected_state = selected_registry.loc[selected_registry['state_id'] == SELECTED_STATE_ID].iloc[0]
selected_hierarchy_dir = Path(selected_state['hierarchy_output_dir'])
selected_metrics_all = pd.read_csv(selected_hierarchy_dir / 'bubble_equivalent_metrics.csv')
selected_edges_all = pd.read_csv(selected_hierarchy_dir / 'bubble_edge_resistance.csv')
selected_nodes_all = pd.read_csv(selected_hierarchy_dir / 'bubble_node_heads.csv')

selected_metrics = selected_metrics_all.loc[selected_metrics_all['compound_bubble_id'] == SELECTED_BUBBLE_ID].copy()
selected_edges = selected_edges_all.loc[selected_edges_all['compound_bubble_id'] == SELECTED_BUBBLE_ID].copy()
selected_nodes = selected_nodes_all.loc[selected_nodes_all['compound_bubble_id'] == SELECTED_BUBBLE_ID].copy()
assert len(selected_metrics) == 1, 'The selected bubble was not uniquely resolved.'
display(selected_metrics)
display(selected_edges.sort_values('K_e', ascending=False).head(20))

,compound_bubble_id,bubble_root_unit_id,upstream_boundary_node,downstream_boundary_node,n_member_units,member_unit_ids,n_nodes,n_edges,solver_success,solver_method,solver_message,solver_nfev,K_eq,G_eq,Q_unit,residual_norm,residual_relative,resistance_law,geometry_method,kb_mode,kb_value,kb_source_method
4,5,15,31,30,14,"3,4,5,6,7,8,9,10,11,12,13,14,15,16",28,41,False,least_squares,The maximum number of function evaluations is ...,4000,1.084128e-07,9.224007e+06,3037.104989,964.193082,0.317471,chezy,wide,based_reference_section,185.975763,based_reference_section


,compound_bubble_id,bubble_root_unit_id,edge_id,u,v,L,w,width_source,k_b,kb_source,d,A,R_h,n,C_z,K_e,G_e,q_unit,dh
56,5,15,37,28,24,470.839162,44.354220,wid_adj_routing,185.975763,default:width-depth ratio,0.238495,10.578243,0.238495,NaN,22.0,0.003716,269.120918,-2.312247,-0.019866
46,5,15,27,17,19,1073.175189,65.462445,wid_adj_routing,185.975763,default:width-depth ratio,0.351994,23.042420,0.351994,NaN,22.0,0.001209,826.867279,14.359663,0.249375
57,5,15,38,28,24,405.090729,59.343533,wid_adj_routing,185.975763,default:width-depth ratio,0.319093,18.936096,0.319093,NaN,22.0,0.000746,1341.096395,-5.161674,-0.019866
66,5,15,48,31,32,1853.137014,91.551814,wid_adj_routing,185.975763,default:width-depth ratio,0.492278,45.068962,0.492278,NaN,22.0,0.000390,2561.961287,49.833812,0.969339
39,5,15,20,14,15,667.103862,76.056219,wid_adj_routing,185.975763,default:width-depth ratio,0.408958,31.103775,0.408958,NaN,22.0,0.000355,2815.953701,-47.844016,-0.812886
64,5,15,45,29,30,1059.633264,83.759803,wid_adj_routing,185.975763,default:width-depth ratio,0.450380,37.723757,0.450380,NaN,22.0,0.000348,2871.891048,53.215608,0.986075
58,5,15,39,25,26,593.272712,85.421328,wid_adj_routing,185.975763,default:width-depth ratio,0.459314,39.235238,0.459314,NaN,22.0,0.000177,5658.776506,71.372751,0.900207
65,5,15,46,32,30,302.170585,82.264976,wid_adj_routing,185.975763,default:width-depth ratio,0.442342,36.389292,0.442342,NaN,22.0,0.000109,9203.817816,16.798776,0.030661
40,5,15,21,22,13,836.646942,106.296683,wid_adj_routing,185.975763,default:width-depth ratio,0.571562,60.755147,0.571562,NaN,22.0,0.000084,11972.963269,-98.082760,-0.803496
34,5,15,15,12,9,1921.146725,125.732144,wid_adj_routing,185.975763,default:width-depth ratio,0.676067,85.003398,0.676067,NaN,22.0,0.000083,12073.032855,14.864929,0.018302


In [22]:
from shapely import wkt

node_positions = {}
for row in selected_nodes.itertuples(index=False):
    point = wkt.loads(row.geometry)
    node_positions[int(row.node_id)] = (point.x, point.y)

fig = go.Figure()
for row in selected_edges.itertuples(index=False):
    u, v = int(row.u), int(row.v)
    if u not in node_positions or v not in node_positions:
        continue
    x0, y0 = node_positions[u]
    x1, y1 = node_positions[v]
    color = '#1f77b4' if row.q_unit >= 0 else '#ff7f0e'
    hover = (
        f'link {row.edge_id}<br>{u} → {v}<br>K_e={row.K_e:.4g}'
        f'<br>q_unit={row.q_unit:.4g}<br>dh={row.dh:.4g}<br>w={row.w:.2f} m<br>L={row.L:.2f} m'
    )
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1], mode='lines', line=dict(color=color, width=2),
        hovertemplate=hover + '<extra></extra>', showlegend=False,
    ))

node_hover = [
    f'node {int(row.node_id)}<br>head={row.head_norm:.6g}<br>residual={row.continuity_residual}'
    for row in selected_nodes.itertuples(index=False)
]
fig.add_trace(go.Scatter(
    x=[node_positions[int(node_id)][0] for node_id in selected_nodes['node_id']],
    y=[node_positions[int(node_id)][1] for node_id in selected_nodes['node_id']],
    mode='markers+text',
    text=selected_nodes['node_id'].astype(int).astype(str),
    textposition='top center',
    marker=dict(
        size=10, color=selected_nodes['head_norm'], colorscale='Viridis', showscale=True,
        colorbar=dict(title='Normalized head'), line=dict(color='black', width=0.5),
    ),
    hovertext=node_hover, hovertemplate='%{hovertext}<extra></extra>', showlegend=False,
))
fig.update_layout(
    title=f'{SELECTED_EXPERIMENT} | {SELECTED_STATE_ID} | bubble {SELECTED_BUBBLE_ID}',
    xaxis=dict(visible=False, scaleanchor='y'), yaxis=dict(visible=False),
    height=700,
)
fig.show()

In [23]:
internal_nodes = selected_nodes.loc[~selected_nodes['is_boundary'].fillna(False).astype(bool)].copy()
internal_nodes['abs_continuity_residual'] = internal_nodes['continuity_residual'].abs()
fig = px.bar(
    internal_nodes.sort_values('abs_continuity_residual', ascending=False),
    x='node_id', y='abs_continuity_residual', log_y=True,
    hover_data=['head_norm', 'continuity_residual', 'head_guess_value'],
    title='Selected bubble: internal-node continuity residuals',
)
fig.show()

## 6. Re-solve the selected bubble from alternative initial heads

This is diagnostic only and does not modify experiment outputs. It tests whether the failed result is primarily sensitive to initialization. The three trials use the saved final heads, a neutral 0.5 internal-head guess, and the original WSE-based guess. If all reach 4,000 evaluations with large residuals, the next investigation should target solver parameterization rather than initialization alone.

In [24]:
if RUN_SELECTED_RESOLVES:
    from hierarchy_level_definition.metrics.equivalent_resistance import solve_bubble_equivalent_K

    metric_row = selected_metrics.iloc[0]
    upstream_node = int(metric_row['upstream_boundary_node'])
    downstream_node = int(metric_row['downstream_boundary_node'])
    solver_edges = selected_edges[['edge_id', 'u', 'v', 'K_e']].copy()
    solver_nodes = selected_nodes[['node_id', 'sword_wse']].copy()

    saved_heads = dict(zip(selected_nodes['node_id'].astype(int), selected_nodes['head_norm']))
    neutral_heads = {int(node_id): 0.5 for node_id in selected_nodes['node_id']}
    neutral_heads[upstream_node] = 1.0
    neutral_heads[downstream_node] = 0.0
    sword_heads = dict(zip(selected_nodes['node_id'].astype(int), selected_nodes['sword_wse']))

    resolve_rows = []
    for label, guess in [
        ('saved_final_heads', saved_heads),
        ('neutral_internal_heads', neutral_heads),
        ('sword_wse_heads', sword_heads),
    ]:
        result = solve_bubble_equivalent_K(
            solver_nodes, solver_edges, upstream_node, downstream_node,
            optional_H_guess=guess, smooth=True,
        )
        resolve_rows.append({
            'initialization': label,
            'solver_success': result['solver_success'],
            'solver_nfev': result['solver_nfev'],
            'residual_norm': result['residual_norm'],
            'residual_relative': result['residual_relative'],
            'Q_unit': result['Q_unit'],
            'K_eq': result['K_eq'],
            'solver_message': result['solver_message'],
        })
    resolve_comparison = pd.DataFrame(resolve_rows)
    display(resolve_comparison)
else:
    print('Selected re-solves disabled. Set RUN_SELECTED_RESOLVES=True to run them.')

,initialization,solver_success,solver_nfev,residual_norm,residual_relative,Q_unit,K_eq,solver_message
0,saved_final_heads,False,4000,9.641365e+02,3.174523e-01,3037.106959,1.084126e-07,The maximum number of function evaluations is ...
1,neutral_internal_heads,True,35,5.200216e-10,1.011094e-12,514.316047,3.780418e-06,`xtol` termination condition is satisfied.
2,sword_wse_heads,False,4000,9.643730e+02,3.178185e-01,3034.351184,1.086096e-07,The maximum number of function evaluations is ...


## 7. Validate terminals from the already assembled bubble subgraph

This proposed production approach does **not** rediscover the unit or rebuild its topology. The structural-unit workflow has already assembled the exact edge set belonging to each simple or compound bubble. Given that edge set, we create its directed subgraph and validate its terminals:

1. The subgraph must be a directed acyclic graph.
2. It must have exactly one node with zero in-degree (the upstream terminal).
3. It must have exactly one node with zero out-degree (the downstream terminal).
4. Every subgraph node must be reachable from the upstream terminal and able to reach the downstream terminal.

The zero-degree rule is therefore applied only after the correct bubble edge set exists. Failure of any validation is an explicit topology/QC error rather than a reason to guess boundaries.

In [25]:
import networkx as nx

def validate_assembled_subgraph_terminals(edge_frame):
    graph = nx.MultiDiGraph()
    graph.add_edges_from(
        (int(row.u), int(row.v), int(row.edge_id))
        for row in edge_frame.itertuples(index=False)
    )
    sources = [node for node in graph if graph.in_degree(node) == 0]
    sinks = [node for node in graph if graph.out_degree(node) == 0]
    is_dag = nx.is_directed_acyclic_graph(graph)
    if not is_dag or len(sources) != 1 or len(sinks) != 1:
        return {
            'valid': False, 'is_dag': is_dag, 'sources': sources, 'sinks': sinks,
            'upstream_node': np.nan, 'downstream_node': np.nan,
            'all_nodes_between_terminals': False,
            'n_nodes': graph.number_of_nodes(), 'n_edges': graph.number_of_edges(),
        }

    upstream_node, downstream_node = sources[0], sinks[0]
    reachable_from_upstream = set(nx.descendants(graph, upstream_node)) | {upstream_node}
    able_to_reach_downstream = set(nx.ancestors(graph, downstream_node)) | {downstream_node}
    nodes_between = reachable_from_upstream & able_to_reach_downstream
    all_nodes_between = nodes_between == set(graph.nodes)
    return {
        'valid': bool(all_nodes_between), 'is_dag': is_dag,
        'sources': sources, 'sinks': sinks,
        'upstream_node': upstream_node, 'downstream_node': downstream_node,
        'all_nodes_between_terminals': bool(all_nodes_between),
        'n_nodes': graph.number_of_nodes(), 'n_edges': graph.number_of_edges(),
    }

terminal_audit_rows = []
for experiment_name in EXPERIMENT_NAMES:
    registry = pd.read_csv(OUTPUT_ROOT / experiment_name / 'state_registry.csv')
    for state_row in registry.itertuples(index=False):
        hierarchy_dir = Path(state_row.hierarchy_output_dir)
        metrics_path = hierarchy_dir / 'bubble_equivalent_metrics.csv'
        edges_path = hierarchy_dir / 'bubble_edge_resistance.csv'
        if not metrics_path.exists() or not edges_path.exists():
            continue
        metrics_frame = pd.read_csv(metrics_path)
        edges_frame = pd.read_csv(edges_path)
        for metric_row in metrics_frame.itertuples(index=False):
            bubble_edges = edges_frame.loc[
                edges_frame['compound_bubble_id'] == metric_row.compound_bubble_id
            ]
            audit = validate_assembled_subgraph_terminals(bubble_edges)
            terminal_audit_rows.append({
                'experiment_name': experiment_name, 'state_id': state_row.state_id,
                'compound_bubble_id': int(metric_row.compound_bubble_id),
                'solver_success': bool(metric_row.solver_success),
                'recorded_upstream_node': int(metric_row.upstream_boundary_node),
                'recorded_downstream_node': int(metric_row.downstream_boundary_node),
                'validated_upstream_node': audit['upstream_node'],
                'validated_downstream_node': audit['downstream_node'],
                'terminal_validation_passed': audit['valid'],
                'is_dag': audit['is_dag'],
                'all_nodes_between_terminals': audit['all_nodes_between_terminals'],
                'source_count': len(audit['sources']), 'sink_count': len(audit['sinks']),
                'n_nodes': audit['n_nodes'], 'n_edges': audit['n_edges'],
            })

terminal_audit = pd.DataFrame(terminal_audit_rows)
terminal_audit['recorded_matches_validated'] = (
    terminal_audit['recorded_upstream_node'].eq(terminal_audit['validated_upstream_node'])
    & terminal_audit['recorded_downstream_node'].eq(terminal_audit['validated_downstream_node'])
)
terminal_audit_summary = (
    terminal_audit.groupby('experiment_name')
    .agg(
        bubble_state_rows=('compound_bubble_id', 'size'),
        valid_assembled_subgraphs=('terminal_validation_passed', 'sum'),
        recorded_boundary_matches=('recorded_matches_validated', 'sum'),
        failed_solves=('solver_success', lambda values: int((~values.astype(bool)).sum())),
    )
    .reset_index()
)
terminal_audit_summary['recorded_boundary_mismatches'] = (
    terminal_audit_summary['bubble_state_rows'] - terminal_audit_summary['recorded_boundary_matches']
)
display(terminal_audit_summary)
display(terminal_audit.loc[~terminal_audit['recorded_matches_validated']].head(100))

,experiment_name,bubble_state_rows,valid_assembled_subgraphs,recorded_boundary_matches,failed_solves,recorded_boundary_mismatches
0,02_compound_bubbles__anchored,31,31,6,1,25
1,02_indep__anchored,161,161,34,11,127
2,03_compound_bubbles__anchored,113,113,78,0,35
3,03_indep__anchored,471,471,318,1,153


,experiment_name,state_id,compound_bubble_id,solver_success,recorded_upstream_node,recorded_downstream_node,validated_upstream_node,validated_downstream_node,terminal_validation_passed,is_dag,all_nodes_between_terminals,source_count,sink_count,n_nodes,n_edges,recorded_matches_validated
0,02_indep__anchored,S000_base,1,True,52,51,52,49,True,True,True,1,1,4,5,False
1,02_indep__anchored,S000_base,2,True,48,42,48,41,True,True,True,1,1,8,11,False
3,02_indep__anchored,S000_base,4,True,38,35,38,33,True,True,True,1,1,6,8,False
4,02_indep__anchored,S000_base,5,False,31,30,31,5,True,True,True,1,1,28,41,False
5,02_indep__anchored,S000_base,6,True,4,3,4,2,True,True,True,1,1,4,5,False
6,02_indep__anchored,S001_unit_2,1,True,50,49,50,47,True,True,True,1,1,4,5,False
7,02_indep__anchored,S001_unit_2,2,True,46,40,46,39,True,True,True,1,1,8,11,False
9,02_indep__anchored,S001_unit_2,4,True,36,33,36,31,True,True,True,1,1,6,8,False
10,02_indep__anchored,S001_unit_2,5,True,29,28,29,3,True,True,True,1,1,28,41,False
12,02_indep__anchored,S002_unit_18,1,True,50,49,50,47,True,True,True,1,1,4,5,False


### Re-solve the currently failed rows with validated subgraph terminals

This remains a read-only diagnostic. It reuses each saved edge resistance and node WSE, changes only the two boundary-node arguments, and compares the result with the stored failed solve. If these solves converge, it demonstrates that the assembled subgraph and resistance data are usable and isolates boundary selection as the production defect.

In [26]:
RUN_FAILED_WITH_VALIDATED_TERMINALS = True

if RUN_FAILED_WITH_VALIDATED_TERMINALS:
    from hierarchy_level_definition.metrics.equivalent_resistance import solve_bubble_equivalent_K

    corrected_rows = []
    failed_audit_rows = terminal_audit.loc[~terminal_audit['solver_success']].copy()
    for audit_row in failed_audit_rows.itertuples(index=False):
        registry = pd.read_csv(OUTPUT_ROOT / audit_row.experiment_name / 'state_registry.csv')
        state_row = registry.loc[registry['state_id'] == audit_row.state_id].iloc[0]
        hierarchy_dir = Path(state_row['hierarchy_output_dir'])
        metrics_frame = pd.read_csv(hierarchy_dir / 'bubble_equivalent_metrics.csv')
        edges_frame = pd.read_csv(hierarchy_dir / 'bubble_edge_resistance.csv')
        nodes_frame = pd.read_csv(hierarchy_dir / 'bubble_node_heads.csv')
        metric = metrics_frame.loc[
            metrics_frame['compound_bubble_id'] == audit_row.compound_bubble_id
        ].iloc[0]
        bubble_edges = edges_frame.loc[
            edges_frame['compound_bubble_id'] == audit_row.compound_bubble_id
        ].copy()
        bubble_nodes = nodes_frame.loc[
            nodes_frame['compound_bubble_id'] == audit_row.compound_bubble_id
        ].copy()
        wse_guess = dict(zip(bubble_nodes['node_id'].astype(int), bubble_nodes['sword_wse']))
        neutral_guess = {int(node_id): 0.5 for node_id in bubble_nodes['node_id']}
        neutral_guess[int(audit_row.validated_upstream_node)] = 1.0
        neutral_guess[int(audit_row.validated_downstream_node)] = 0.0
        corrected_wse = solve_bubble_equivalent_K(
            bubble_nodes[['node_id', 'sword_wse']],
            bubble_edges[['edge_id', 'u', 'v', 'K_e']],
            int(audit_row.validated_upstream_node),
            int(audit_row.validated_downstream_node),
            optional_H_guess=wse_guess, smooth=True,
        )
        corrected_neutral = solve_bubble_equivalent_K(
            bubble_nodes[['node_id', 'sword_wse']],
            bubble_edges[['edge_id', 'u', 'v', 'K_e']],
            int(audit_row.validated_upstream_node),
            int(audit_row.validated_downstream_node),
            optional_H_guess=neutral_guess, smooth=True,
        )
        selected_initialization = (
            'wse' if corrected_wse['solver_success'] else 'neutral_fallback'
        )
        corrected = (
            corrected_wse if corrected_wse['solver_success'] else corrected_neutral
        )
        corrected_rows.append({
            'experiment_name': audit_row.experiment_name, 'state_id': audit_row.state_id,
            'compound_bubble_id': audit_row.compound_bubble_id,
            'recorded_terminals': f'{audit_row.recorded_upstream_node} → {audit_row.recorded_downstream_node}',
            'validated_terminals': f'{int(audit_row.validated_upstream_node)} → {int(audit_row.validated_downstream_node)}',
            'stored_success': bool(metric['solver_success']),
            'wse_success': corrected_wse['solver_success'],
            'neutral_success': corrected_neutral['solver_success'],
            'selected_initialization': selected_initialization,
            'corrected_success': corrected['solver_success'],
            'stored_nfev': int(metric['solver_nfev']),
            'wse_nfev': corrected_wse['solver_nfev'],
            'neutral_nfev': corrected_neutral['solver_nfev'],
            'corrected_nfev': corrected['solver_nfev'],
            'stored_residual_relative': metric['residual_relative'],
            'corrected_residual_relative': corrected['residual_relative'],
            'stored_K_eq': metric['K_eq'], 'corrected_K_eq': corrected['K_eq'],
        })
    corrected_failed_solves = pd.DataFrame(corrected_rows)
    display(corrected_failed_solves)
    print(
        f"Corrected solves successful: {int(corrected_failed_solves['corrected_success'].sum())}"
        f" / {len(corrected_failed_solves)}"
    )
else:
    print('Validated-terminal re-solves disabled.')

,experiment_name,state_id,compound_bubble_id,recorded_terminals,validated_terminals,stored_success,wse_success,neutral_success,selected_initialization,corrected_success,stored_nfev,wse_nfev,neutral_nfev,corrected_nfev,stored_residual_relative,corrected_residual_relative,stored_K_eq,corrected_K_eq
0,02_indep__anchored,S000_base,5,31 → 30,31 → 5,False,True,True,wse,True,4000,23,31,23,0.317471,1.856787e-15,1.084128e-07,6.222240e-09
1,02_indep__anchored,S009_unit_11,5,29 → 28,29 → 5,False,True,True,wse,True,4000,32,41,32,0.157857,8.331340e-16,7.234279e-07,5.831053e-09
2,02_indep__anchored,S010_unit_9,5,29 → 28,29 → 5,False,True,True,wse,True,4000,23,30,23,0.300177,2.380376e-15,9.645829e-08,6.217800e-09
3,02_indep__anchored,S011_unit_13,5,29 → 28,29 → 5,False,True,True,wse,True,4000,26,33,26,0.317602,3.943055e-15,1.085600e-07,6.217238e-09
4,02_indep__anchored,S014_unit_3,5,29 → 28,29 → 5,False,True,True,wse,True,4000,22,30,22,0.295993,1.091295e-15,1.121667e-07,4.710486e-09
5,02_indep__anchored,S016_unit_7,5,27 → 26,27 → 5,False,True,True,wse,True,4000,49,74,49,0.131200,5.757645e-15,7.935796e-07,4.927544e-09
6,02_indep__anchored,S020_unit_8,5,29 → 28,29 → 5,False,False,True,neutral_fallback,True,4000,4000,72,72,0.271316,3.092921e-15,6.518600e-08,6.112762e-09
7,02_indep__anchored,S021_unit_15,5,28 → 19,28 → 5,False,True,True,wse,True,4000,23,29,23,0.217942,6.839932e-16,7.745016e-08,6.221188e-09
8,02_indep__anchored,S023_unit_12,5,29 → 28,29 → 5,False,True,True,wse,True,4000,25,31,25,0.316702,2.516521e-14,1.078858e-07,6.475433e-09
9,02_indep__anchored,S024_unit_14,5,29 → 28,29 → 5,False,True,True,wse,True,4000,19,28,19,0.144202,1.995764e-15,2.403507e-08,5.994091e-09


Corrected solves successful: 13 / 13


## 8. Test the hydraulic-domain selection rule on a selected unit

A structural unit is eligible for its own two-terminal hydraulic solve only when its selected edge set is closed to lateral exchange: external edges may enter at the source and leave at the sink, but no excluded edge may enter the sink, leave the source, or touch an internal node. If that test fails, the primary evaluation domain becomes the containing compound context.

For a member-unit collapse, the base compound's validated outer source and sink are mapped into the child. The child evaluation domain then contains **every directed link on a path between those matched anchors**. This preserves one physical control volume while allowing all internal flow splits and rejoins.

In [39]:
DOMAIN_TEST_EXPERIMENT = '03_indep__anchored'
DOMAIN_TEST_BASE_STATE = 'S000_base'
DOMAIN_TEST_CHILD_STATE = 'S027_unit_24'
DOMAIN_TEST_BASE_UNIT_ID = 24

domain_experiment_dir = OUTPUT_ROOT / DOMAIN_TEST_EXPERIMENT
domain_registry = pd.read_csv(domain_experiment_dir / 'state_registry.csv')
domain_base_row = domain_registry.loc[domain_registry['state_id'] == DOMAIN_TEST_BASE_STATE].iloc[0]
domain_child_row = domain_registry.loc[domain_registry['state_id'] == DOMAIN_TEST_CHILD_STATE].iloc[0]
domain_base_links = pd.DataFrame(__import__('geopandas').read_file(domain_base_row['analysis_links_path']))
domain_child_links = pd.DataFrame(__import__('geopandas').read_file(domain_child_row['analysis_links_path']))
domain_base_nodes = pd.DataFrame(__import__('geopandas').read_file(domain_base_row['directed_nodes_path']))
domain_child_nodes = pd.DataFrame(__import__('geopandas').read_file(domain_child_row['directed_nodes_path']))


In [40]:
def extract_complete_directed_domain(full_links, source_node, sink_node):
    graph = nx.DiGraph()
    graph.add_edges_from(
        (int(row.id_us_node), int(row.id_ds_node))
        for row in full_links.itertuples(index=False)
    )
    source_node, sink_node = int(source_node), int(sink_node)
    if not graph.has_node(source_node) or not graph.has_node(sink_node):
        raise ValueError('Evaluation source or sink is absent from the network.')
    if not nx.has_path(graph, source_node, sink_node):
        raise ValueError(f'No directed path exists from {source_node} to {sink_node}.')
    nodes = (
        (set(nx.descendants(graph, source_node)) | {source_node})
        & (set(nx.ancestors(graph, sink_node)) | {sink_node})
    )
    return full_links.loc[
        full_links['id_us_node'].astype(int).isin(nodes)
        & full_links['id_ds_node'].astype(int).isin(nodes)
    ].copy()

def audit_hydraulic_domain_closure(domain_links, full_links, source_node, sink_node):
    source_node, sink_node = int(source_node), int(sink_node)
    domain_link_ids = set(domain_links['id_link'].astype(int))
    domain_nodes = (
        set(domain_links['id_us_node'].astype(int))
        | set(domain_links['id_ds_node'].astype(int))
    )
    internal_nodes = domain_nodes - {source_node, sink_node}
    violations = []
    for row in full_links.loc[~full_links['id_link'].astype(int).isin(domain_link_ids)].itertuples(index=False):
        link_id, u, v = int(row.id_link), int(row.id_us_node), int(row.id_ds_node)
        reason = None
        if u in internal_nodes or v in internal_nodes:
            reason = 'external_edge_touches_internal_node'
        elif u == source_node:
            reason = 'external_outflow_from_source'
        elif v == sink_node:
            reason = 'external_inflow_to_sink'
        if reason is not None:
            violations.append({'id_link': link_id, 'id_us_node': u, 'id_ds_node': v, 'reason': reason})
    return {
        'is_closed': len(violations) == 0,
        'source_node': source_node, 'sink_node': sink_node,
        'n_links': len(domain_link_ids), 'n_nodes': len(domain_nodes),
        'violations': pd.DataFrame(violations),
    }

def stable_child_node_for_parent(node_match, parent_node):
    candidates = node_match.loc[
        pd.to_numeric(node_match['matched_parent_node_id'], errors='coerce').eq(int(parent_node))
        & node_match['match_within_tolerance'].fillna(False).astype(bool)
    ].copy()
    if candidates.empty:
        raise ValueError(f'No stable child match was found for parent node {parent_node}.')
    candidates = candidates.sort_values(['match_distance', 'child_id_node'])
    return int(candidates.iloc[0]['child_id_node'])


### First test: which base structural units are hydraulically closed?

This uses each unit's existing membership list; it does not redetect units. A failed closure test means the structural unit is still valid for collapse selection, but its isolated resistance cannot represent the embedded network because excluded links exchange flow at its internal or boundary nodes.

In [41]:
base_hierarchy_dir = Path(domain_base_row['hierarchy_output_dir'])
base_variant_dir = Path(domain_base_row['variant_output_dir'])
base_unit_summary = pd.read_csv(base_hierarchy_dir / 'unit_summary.csv')
base_unit_membership = pd.read_csv(base_variant_dir / 'summary' / 'unit_link_membership.csv')

unit_closure_rows = []
for unit_row in base_unit_summary.itertuples(index=False):
    member_link_ids = set(
        base_unit_membership.loc[
            base_unit_membership['unit_id'].astype(int) == int(unit_row.unit_id), 'id_link'
        ].astype(int)
    )
    unit_links = domain_base_links.loc[domain_base_links['id_link'].astype(int).isin(member_link_ids)].copy()
    closure = audit_hydraulic_domain_closure(
        unit_links, domain_base_links, int(unit_row.bifurcation), int(unit_row.confluence)
    )
    unit_closure_rows.append({
        'unit_id': int(unit_row.unit_id),
        'source_node': int(unit_row.bifurcation), 'sink_node': int(unit_row.confluence),
        'compound_bubble_id': unit_row.compound_bubble_id,
        'in_compound_bubble': bool(unit_row.in_compound_bubble),
        'n_links': closure['n_links'], 'n_nodes': closure['n_nodes'],
        'is_hydraulically_closed': closure['is_closed'],
        'violating_link_ids': ','.join(map(str, closure['violations'].get('id_link', []))),
        'violation_count': len(closure['violations']),
    })
unit_closure_audit = pd.DataFrame(unit_closure_rows)
display(unit_closure_audit.groupby(['in_compound_bubble', 'is_hydraulically_closed']).size().rename('n_units').reset_index())
display(unit_closure_audit.loc[unit_closure_audit['unit_id'] == DOMAIN_TEST_BASE_UNIT_ID])

,in_compound_bubble,is_hydraulically_closed,n_units
0,False,True,8
1,True,False,16
2,True,True,7


,unit_id,source_node,sink_node,compound_bubble_id,in_compound_bubble,n_links,n_nodes,is_hydraulically_closed,violating_link_ids,violation_count
22,23,44,50,14,True,5,5,False,"73,64,80",3


In [42]:
selected_unit = base_unit_summary.loc[base_unit_summary['unit_id'] == DOMAIN_TEST_BASE_UNIT_ID].iloc[0]
selected_closure = unit_closure_audit.loc[unit_closure_audit['unit_id'] == DOMAIN_TEST_BASE_UNIT_ID].iloc[0]
selected_member_link_ids = set(
    base_unit_membership.loc[
        base_unit_membership['unit_id'].astype(int) == DOMAIN_TEST_BASE_UNIT_ID, 'id_link'
    ].astype(int)
)
selected_unit_links = domain_base_links.loc[
    domain_base_links['id_link'].astype(int).isin(selected_member_link_ids)
].copy()

if bool(selected_closure.is_hydraulically_closed):
    selected_compound_id = None
    base_domain_name = 'standalone_unit'
    child_domain_name = 'matched_standalone_unit'
    base_source = int(selected_unit['bifurcation'])
    base_sink = int(selected_unit['confluence'])
    base_evaluation_links = selected_unit_links
else:
    selected_compound_id = int(selected_unit['compound_bubble_id'])
    base_domain_name = 'containing_compound_context'
    child_domain_name = 'matched_compound_context'
    base_bubble_edges = pd.read_csv(base_hierarchy_dir / 'bubble_edge_resistance.csv')
    base_bubble_edges = base_bubble_edges.loc[
        base_bubble_edges['compound_bubble_id'] == selected_compound_id
    ].copy()
    base_terminal_validation = validate_assembled_subgraph_terminals(base_bubble_edges)
    assert base_terminal_validation['valid'], base_terminal_validation
    base_source = int(base_terminal_validation['upstream_node'])
    base_sink = int(base_terminal_validation['downstream_node'])
    base_evaluation_links = extract_complete_directed_domain(
        domain_base_links, base_source, base_sink
    )
base_closure = audit_hydraulic_domain_closure(
    base_evaluation_links, domain_base_links, base_source, base_sink
)

child_node_match = pd.read_csv(Path(domain_child_row['variant_output_dir']) / 'matching' / 'node_match.csv')
child_source = stable_child_node_for_parent(child_node_match, base_source)
child_sink = stable_child_node_for_parent(child_node_match, base_sink)
child_evaluation_links = extract_complete_directed_domain(
    domain_child_links, child_source, child_sink
)
child_closure = audit_hydraulic_domain_closure(
    child_evaluation_links, domain_child_links, child_source, child_sink
)

domain_selection_summary = pd.DataFrame([
    {
        'state': 'base', 'evaluation_domain': base_domain_name,
        'source_node': base_source, 'sink_node': base_sink,
        'n_links': len(base_evaluation_links), 'is_closed': base_closure['is_closed'],
    },
    {
        'state': 'child', 'evaluation_domain': child_domain_name,
        'source_node': child_source, 'sink_node': child_sink,
        'n_links': len(child_evaluation_links), 'is_closed': child_closure['is_closed'],
    },
])
print(f'Base unit {DOMAIN_TEST_BASE_UNIT_ID} closed: {bool(selected_closure.is_hydraulically_closed)}')
print(f'Selected evaluation domain: {base_domain_name}')
if selected_compound_id is not None:
    print(f'Containing compound: {selected_compound_id}')
display(domain_selection_summary)
assert base_closure['is_closed'] and child_closure['is_closed']

Base unit 23 closed: False
Selected evaluation domain: containing_compound_context
Containing compound: 14


,state,evaluation_domain,source_node,sink_node,n_links,is_closed
0,base,containing_compound_context,44,54,23,True
1,child,matched_compound_context,44,52,23,True


### Solve every link in the selected base and child domains

Every included link receives its own geometry-derived resistance. The network solve then uses all edge resistances simultaneously, enforces continuity at every internal node, and derives one source-to-sink equivalent resistance from the total source inflow.

In [43]:
from hierarchy_level_definition.metrics import BubbleEquivalentResistanceConfig
from hierarchy_level_definition.metrics.equivalent_resistance import solve_bubble_equivalent_K
from hierarchy_level_definition.metrics.bubble_equivalent_metrics import prepare_bubble_resistance_edges
from network_variants.equivalent_resistance_experiment import _compute_equivalent_resistance

domain_manifest = json.loads((domain_experiment_dir / 'experiment_manifest.json').read_text())
domain_hierarchy_options = domain_manifest['options']['hierarchy']
domain_resistance_options = domain_hierarchy_options['equivalent_resistance_config']
domain_config = BubbleEquivalentResistanceConfig(
    resistance_law=domain_resistance_options['resistance_law'],
    geometry_method=domain_resistance_options['geometry_method'],
    kb_mode=domain_resistance_options['kb_mode'],
    reference_kb_summary=domain_hierarchy_options['equivalent_reference_kb_summary'],
    length_field_candidates=tuple(domain_resistance_options['length_field_candidates']),
    default_chezy_cz=domain_resistance_options['default_chezy_cz'],
)

def solve_domain_with_neutral_fallback(links, nodes, source_node, sink_node, config):
    primary_result, resolved_kb = _compute_equivalent_resistance(
        links, nodes, upstream_node=source_node, downstream_node=sink_node, config=config,
    )
    primary_result = dict(primary_result)
    primary_result['initialization_used'] = 'available_node_heads'
    primary_result['primary_solver_success'] = bool(primary_result['solver_success'])
    primary_result['primary_residual_relative'] = primary_result['residual_relative']
    if primary_result['solver_success']:
        return primary_result, resolved_kb

    prepared_edges, fallback_kb = prepare_bubble_resistance_edges(links, config=config)
    incident_ids = set(prepared_edges['u']) | set(prepared_edges['v'])
    node_subset = nodes.loc[pd.to_numeric(nodes['id_node'], errors='coerce').isin(incident_ids)]
    neutral_guess = {int(node_id): 0.5 for node_id in incident_ids}
    neutral_guess[int(source_node)] = 1.0
    neutral_guess[int(sink_node)] = 0.0
    fallback_result = solve_bubble_equivalent_K(
        node_subset, prepared_edges,
        upstream_node=int(source_node), downstream_node=int(sink_node),
        optional_H_guess=neutral_guess, smooth=config.smooth, eps=config.smooth_eps,
    )
    fallback_result = dict(fallback_result)
    fallback_result['initialization_used'] = 'neutral_internal_heads_fallback'
    fallback_result['primary_solver_success'] = False
    fallback_result['primary_residual_relative'] = primary_result['residual_relative']
    return fallback_result, fallback_kb.to_dict()

base_node_ids = set(base_evaluation_links['id_us_node'].astype(int)) | set(base_evaluation_links['id_ds_node'].astype(int))
child_node_ids = set(child_evaluation_links['id_us_node'].astype(int)) | set(child_evaluation_links['id_ds_node'].astype(int))
base_result, _ = solve_domain_with_neutral_fallback(
    base_evaluation_links,
    domain_base_nodes.loc[domain_base_nodes['id_node'].astype(int).isin(base_node_ids)],
    base_source, base_sink, domain_config,
)
child_result, _ = solve_domain_with_neutral_fallback(
    child_evaluation_links,
    domain_child_nodes.loc[domain_child_nodes['id_node'].astype(int).isin(child_node_ids)],
    child_source, child_sink, domain_config,
)

domain_result = pd.DataFrame([
    {
        'state': 'base', 'source_node': base_source, 'sink_node': base_sink,
        'n_links': len(base_evaluation_links),
        'initialization_used': base_result['initialization_used'],
        'primary_solver_success': base_result['primary_solver_success'],
        'primary_residual_relative': base_result['primary_residual_relative'],
        'solver_success': base_result['solver_success'],
        'solver_nfev': base_result['solver_nfev'],
        'residual_relative': base_result['residual_relative'], 'K_eq': base_result['K_eq'],
    },
    {
        'state': 'child', 'source_node': child_source, 'sink_node': child_sink,
        'n_links': len(child_evaluation_links),
        'initialization_used': child_result['initialization_used'],
        'primary_solver_success': child_result['primary_solver_success'],
        'primary_residual_relative': child_result['primary_residual_relative'],
        'solver_success': child_result['solver_success'],
        'solver_nfev': child_result['solver_nfev'],
        'residual_relative': child_result['residual_relative'], 'K_eq': child_result['K_eq'],
    },
])
lambda_k = child_result['K_eq'] / base_result['K_eq']
display(domain_result)
print(f'Lambda_K = {lambda_k:.6f}')
print(f'Equivalent-resistance change = {(lambda_k - 1.0) * 100.0:.3f}%')
assert bool(domain_result['solver_success'].all())

,state,source_node,sink_node,n_links,solver_success,solver_nfev,residual_relative,K_eq
0,base,44,54,23,True,38,1.891371e-15,4.341091e-07
1,child,44,52,23,False,4000,2.730780e-01,8.702541e-07


Lambda_K = 2.004690
Equivalent-resistance change = 100.469%


AssertionError: 

In [32]:
base_link_audit = base_evaluation_links[['id_link', 'id_us_node', 'id_ds_node', 'len', 'wid_adj_routing']].copy()
base_link_audit.insert(0, 'state', 'base')
child_link_audit = child_evaluation_links[['id_link', 'id_us_node', 'id_ds_node', 'len', 'wid_adj_routing']].copy()
child_link_audit.insert(0, 'state', 'child')
display(pd.concat([base_link_audit, child_link_audit], ignore_index=True).sort_values(['state', 'id_link']))

,state,id_link,id_us_node,id_ds_node,len,wid_adj_routing
7,base,64,43,45,746.385277,284.587289
14,base,65,46,47,678.220666,165.607086
0,base,66,44,43,559.191788,387.198528
2,base,67,43,50,453.257364,196.996558
1,base,68,44,48,199.783427,166.095895
11,base,69,45,46,37.493884,226.207592
10,base,70,45,51,365.289436,97.804984
17,base,71,46,47,640.726782,126.263605
21,base,72,47,49,250.533400,330.090791
3,base,73,48,56,908.674820,75.980785


## 9. Decision checklist

Before changing the production solver, record: 

1. Whether failures are limited to large or highly connected bubbles.
2. Whether alternative initial heads materially reduce the residual.
3. Which internal nodes dominate the remaining continuity error.
4. Whether the resistance or flow range spans many orders of magnitude.
5. Whether affected transition rows should be excluded temporarily or recomputed after a solver change.

Do not treat a finite `K_eq` as a successful result when `solver_success` is false.